# CALCUALTE AGE AND OBSERVATION PERIOD 

The script calculates both the age and the observation period for all individuals in the AoU. Age is defined as the age at the most recent electronic health record encounter, while the observation period is measured from the first to the last electronic health record. The duration in days is then converted to years by dividing by 365.25. 

In [1]:
# Import the packages
import os
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
from functools import reduce

In [2]:
# Retrieve the value of the environment variable 'WORKSPACE_CDR' and assign it to the variable 'dataset'.
dataset = os.getenv("WORKSPACE_CDR")
# Display the value of 'dataset'
dataset

'fc-aou-cdr-prod-ct.C2024Q3R9'

In [3]:
import pandas
import os

# This query represents dataset "data_of_birth_ehr_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_73356962_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

dataset_73356962_person_df = pandas.read_gbq(
    dataset_73356962_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_73356962_person_df.head(5)

Downloading:   0%|          | 0/393601 [00:00<?, ?rows/s]

,person_id,date_of_birth
0,1581351,2003-06-15 00:00:00+00:00
1,4521223,2003-06-15 00:00:00+00:00
2,1691808,1931-06-15 00:00:00+00:00
3,6152588,2001-06-15 00:00:00+00:00
4,2922960,1937-06-15 00:00:00+00:00


In [11]:

query = f"""                                
SELECT *
FROM `{dataset}.observation_period`
LIMIT 10
""" 
    
# Run the query for each person_id
observation_period = pd.read_gbq(query, dialect="standard")


In [23]:
query = f"""
SELECT person_id, COUNT(person_id) AS person_count
FROM `{dataset}.observation_period`
GROUP BY person_id
"""

# Run the query and load the result into a DataFrame
observation_period = pd.read_gbq(query, dialect="standard")


In [26]:
observation_period["person_count"].value_counts()

person_count
1    633547
Name: count, dtype: Int64

In [18]:
cohort = dataset_73356962_person_df["person_id"]

In [19]:
batch_results = [] # Initialize an empty list to store results
chunk_size = 1000

# Loop through chunks
for i in range(0, len(cohort), chunk_size):
    cohort_chunk = cohort[i:i + chunk_size]
    # Loop through each person_id in the cohort
    query = f"""
    SELECT person_id, observation_period_start_date, observation_period_end_date
    FROM `{dataset}.observation_period`
    WHERE person_id IN {tuple(cohort_chunk)}
    """ 
    # Run the query for each person_id
    batch_result = pd.read_gbq(query, dialect="standard")
    batch_results.append(batch_result)

    
# Combine results from all queries
observation_period = pd.concat(batch_results, ignore_index=True)  # Added ignore_index to reset index


In [32]:
observation_period.to_csv("observation_period.tsv", sep ='\t', index=False)

In [35]:
dataset_73356962_person_df.to_csv("date_of_birth.tsv", sep ='\t', index=False)

In [40]:
!gsutil cp observation_period.tsv {my_bucket}/data/phenotype/covariates/
!gsutil cp date_of_birth.tsv {my_bucket}/data/phenotype/covariates/

Copying file://observation_period.tsv [Content-Type=text/tab-separated-values]...
/ [1 files][ 11.3 MiB/ 11.3 MiB]                                                
Operation completed over 1 objects/11.3 MiB.                                     
Copying file://date_of_birth.tsv [Content-Type=text/tab-separated-values]...
\ [1 files][ 12.8 MiB/ 12.8 MiB]                                                
Operation completed over 1 objects/12.8 MiB.                                     


In [41]:
!gsutil ls {my_bucket}/data/phenotype/covariates/

gs://fc-secure-2f51529a-432d-4d4a-806c-64c071b8ae57/data/phenotype/covariates/date_of_birth.tsv
gs://fc-secure-2f51529a-432d-4d4a-806c-64c071b8ae57/data/phenotype/covariates/observation_period.tsv


In [48]:
observation_age_period = (
observation_period.merge(dataset_73356962_person_df, 
                         on ="person_id", 
                         how ="inner")
)

In [69]:
# Convert both columns to datetime and remove any timezone information (make them naive)
observation_age_period["observation_period_end_date"] = (
    pd.to_datetime(observation_age_period["observation_period_end_date"]).dt.tz_localize(None)
)

observation_age_period["observation_period_start_date"] = (
    pd.to_datetime(observation_age_period["observation_period_start_date"]).dt.tz_localize(None)
)


observation_age_period["date_of_birth"] = (
    pd.to_datetime(observation_age_period["date_of_birth"]).dt.tz_localize(None)
)


In [73]:
observation_age_period["age"] = (
(observation_age_period["observation_period_end_date"] - 
 observation_age_period["date_of_birth"]).dt.days / 365.25
)

In [72]:
observation_age_period["observation_period"] = (
(observation_age_period["observation_period_end_date"] - 
 observation_age_period["observation_period_start_date"]).dt.days / 365.25
)

In [80]:
observation_age_period.to_csv("observation_age_period.tsv", sep ='\t',  index=False)

In [83]:
!gsutil cp observation_age_period.tsv {my_bucket}/data/phenotype/covariates/

Copying file://observation_age_period.tsv [Content-Type=text/tab-separated-values]...
| [1 files][ 29.0 MiB/ 29.0 MiB]    1.5 MiB/s                                   
Operation completed over 1 objects/29.0 MiB.                                     


In [84]:
!gsutil ls {my_bucket}/data/phenotype/covariates/

gs://fc-secure-2f51529a-432d-4d4a-806c-64c071b8ae57/data/phenotype/covariates/date_of_birth.tsv
gs://fc-secure-2f51529a-432d-4d4a-806c-64c071b8ae57/data/phenotype/covariates/observation_age_period.tsv
gs://fc-secure-2f51529a-432d-4d4a-806c-64c071b8ae57/data/phenotype/covariates/observation_period.tsv
